# 📊 Semana 19-20: Manipulación Avanzada y Visualización de Datos

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Avanzado

---

## 📋 ¿Qué vas a aprender esta semana?

| Bloque | Temas |
|--------|-------|
| 🐼 Pandas avanzado | `merge`, `melt`/`pivot`, `resample`, `apply`, ventanas |
| 🔢 NumPy avanzado | SVD, álgebra lineal, simulaciones, broadcasting complejo |
| 📈 Matplotlib | Subplots, estilos, anotaciones, gráficos compuestos |
| 🎨 Seaborn | Gráficos estadísticos, FacetGrid, correlaciones, distribuciones |

---

> 💡 **Prerequisito:** Semanas 1-18 completadas.  
> **Instalaciones necesarias:**
> ```bash
> pip install pandas numpy matplotlib seaborn
> ```

---
## 🐼 Bloque 1: Pandas Avanzado

### 📘 Teoría

#### Merges — combinar DataFrames
```python
# Inner join (solo coincidencias)
pd.merge(df1, df2, on='id', how='inner')

# Left join (todas las filas de df1)
pd.merge(df1, df2, on='id', how='left')

# Merge con claves distintas
pd.merge(ventas, clientes, left_on='cliente_id', right_on='id')

# Merge por índice
df1.join(df2, how='left')
```

#### Melt y Pivot — reestructurar datos
```python
# Wide → Long (melt): útil para graficar y analizar
df_long = df.melt(id_vars=['fecha','region'],
                  value_vars=['ventas_online','ventas_tienda'],
                  var_name='canal', value_name='monto')

# Long → Wide (pivot): útil para tablas resumen
df_wide = df_long.pivot_table(index='fecha', columns='canal',
                               values='monto', aggfunc='sum')
```

#### Resample — series de tiempo
```python
df.set_index('fecha').resample('M')['ventas'].sum()  # mensual
df.set_index('fecha').resample('Q')['ventas'].mean() # trimestral
df.set_index('fecha').resample('W')['ventas'].agg(['sum','mean','count'])
```

#### Ventanas deslizantes
```python
df['media_movil_7d']   = df['ventas'].rolling(7).mean()
df['media_movil_30d']  = df['ventas'].rolling(30).mean()
df['expansion_acum']   = df['ventas'].expanding().sum()
df['pct_cambio']       = df['ventas'].pct_change()
```

### 💡 Ejemplo resuelto 1 — Análisis completo de ventas con pandas avanzado

In [ ]:
import pandas as pd
import numpy as np

# ✅ Pipeline completo usando técnicas avanzadas de pandas

np.random.seed(42)

# ── Crear 3 tablas relacionadas ──
n_ventas = 1000
fechas   = pd.date_range('2023-01-01', '2024-12-31', periods=n_ventas)
regiones = ['AMBA', 'Centro', 'NOA', 'Cuyo', 'Patagonia']
canales  = ['Online', 'Tienda', 'Telefono']
cats     = ['Electrónica', 'Ropa', 'Hogar', 'Deportes']

df_ventas = pd.DataFrame({
    'fecha':      fechas,
    'venta_id':   range(1, n_ventas+1),
    'cliente_id': np.random.randint(1, 201, n_ventas),
    'producto_id':np.random.randint(1, 51, n_ventas),
    'region':     np.random.choice(regiones, n_ventas),
    'canal':      np.random.choice(canales, n_ventas, p=[0.5,0.35,0.15]),
    'cantidad':   np.random.randint(1, 8, n_ventas),
    'precio':     np.round(np.random.uniform(100, 5000, n_ventas), 2),
})
df_ventas['ingreso'] = df_ventas['cantidad'] * df_ventas['precio']

df_clientes = pd.DataFrame({
    'cliente_id': range(1, 201),
    'nombre':     [f'Cliente_{i}' for i in range(1, 201)],
    'segmento':   np.random.choice(['Premium','Standard','Basico'], 200, p=[0.2,0.5,0.3]),
    'ciudad':     np.random.choice(regiones, 200),
})

df_productos = pd.DataFrame({
    'producto_id': range(1, 51),
    'nombre':      [f'Producto_{i}' for i in range(1, 51)],
    'categoria':   np.random.choice(cats, 50),
    'costo':       np.round(np.random.uniform(50, 2000, 50), 2),
})

# ── MERGE: enriquecer ventas con info de clientes y productos ──
df_full = (
    df_ventas
    .merge(df_clientes, on='cliente_id', how='left')
    .merge(df_productos, on='producto_id', how='left')
)
df_full['margen'] = df_full['ingreso'] - df_full['cantidad'] * df_full['costo']

print('=== MERGE ===')
print(f'  Ventas: {len(df_ventas)} | Full: {len(df_full)} | Columnas: {len(df_full.columns)}')
print(f'  Ingreso total: ${df_full["ingreso"].sum():,.0f}')
print(f'  Margen total:  ${df_full["margen"].sum():,.0f}')

# ── MELT: convertir canales de wide a long ──
df_canal_region = (
    df_full.groupby(['region','canal'])['ingreso']
    .sum()
    .reset_index()
)
df_wide = df_canal_region.pivot_table(
    index='region', columns='canal', values='ingreso', aggfunc='sum', fill_value=0
).round(0)
print('\n=== PIVOT TABLE — Ingreso por región y canal ===')
print(df_wide.to_string())

# De vuelta a long con melt
df_long = df_wide.reset_index().melt(
    id_vars='region', var_name='canal', value_name='ingreso'
)
print(f'\n  Melt: {len(df_wide)} filas wide → {len(df_long)} filas long')

# ── RESAMPLE: tendencia mensual ──
print('\n=== RESAMPLE — Tendencia mensual de ingreso ===')
mensual = (
    df_full.set_index('fecha')['ingreso']
    .resample('M')
    .agg(['sum','mean','count'])
    .rename(columns={'sum':'total','mean':'promedio','count':'transacciones'})
)
for mes, row in mensual.iterrows():
    barra = '█' * int(row['total'] / mensual['total'].max() * 25)
    print(f'  {mes.strftime("%Y-%m")}  {barra:25}  ${row["total"]:>10,.0f}  ({row["transacciones"]:>3} txns)')

# ── VENTANAS DESLIZANTES ──
print('\n=== VENTANAS — Media móvil 30 días ===')
diario = df_full.set_index('fecha')['ingreso'].resample('D').sum()
ma30   = diario.rolling(30).mean()
pct_cambio = diario.pct_change(30)  # vs 30 días atrás

print(f'  Últimos 5 días:')
for fecha in diario.index[-5:]:
    d   = diario.get(fecha, 0)
    ma  = ma30.get(fecha, 0)
    pct = pct_cambio.get(fecha, 0)
    if pd.notna(ma) and pd.notna(pct):
        tendencia = '↑' if pct > 0 else '↓'
        print(f'    {fecha.date()}  diario=${d:>8,.0f}  MA30=${ma:>8,.0f}  {tendencia}{abs(pct)*100:.1f}%')

# ── APPLY con función compleja ──
print('\n=== APPLY — Clasificar margen por venta ===')

def clasificar_margen(row):
    pct = row['margen'] / row['ingreso'] if row['ingreso'] > 0 else 0
    if pct > 0.5:  return 'Alto'
    elif pct > 0.2: return 'Medio'
    else:           return 'Bajo'

df_full['clase_margen'] = df_full.apply(clasificar_margen, axis=1)
print(df_full['clase_margen'].value_counts().to_string())

# ── GROUPBY avanzado con múltiples métricas ──
print('\n=== Performance por segmento y canal ===')
perf = df_full.groupby(['segmento','canal']).agg(
    total_ing   = ('ingreso','sum'),
    avg_ticket  = ('ingreso','mean'),
    transacciones=('venta_id','count'),
    margen_pct  = ('margen', lambda x: (x.sum() / df_full.loc[x.index,'ingreso'].sum() * 100).round(1))
).round(1)
print(perf.sort_values('total_ing', ascending=False).head(9).to_string())

### ✏️ Ejercicio guiado 1 — Series de tiempo con resample y ventanas

Usando `df_full` del ejemplo anterior, realizá estos análisis de series de tiempo.

**Pistas:**
- `resample('Q')` agrupa por trimestre
- `.rolling(window).agg({'col': ['min','max','mean']})` aplica múltiples funciones
- `shift(n)` desplaza la serie n posiciones — útil para calcular variaciones YoY

In [ ]:
# (Requiere df_full del ejemplo anterior)

# ✏️ Análisis 1: Ingreso trimestral con variación vs trimestre anterior
print('=== Ingreso trimestral ===')
# trimestral = df_full.set_index('fecha')['ingreso'].resample('Q').sum()
# variacion  = trimestral.pct_change() * 100
# Tu código aquí:


# ✏️ Análisis 2: Volatilidad semanal (std de ingreso diario en cada semana)
print('\n=== Volatilidad semanal ===')
# Pista: resample('W') + std()


# ✏️ Análisis 3: Clientes que compraron en 2023 pero NO en 2024 (churn)
print('\n=== Clientes en churn ===')
# Pista: filtrar por año, obtener sets de cliente_id, restar conjuntos


# ✏️ Análisis 4: Ticket promedio con ventana de 7 días
print('\n=== Ticket promedio MA7 (últimos 10 días) ===')
# Pista: resample('D') + rolling(7) sobre precio promedio diario


<details>
<summary>👀 Ver solución</summary>

```python
# Análisis 1
trimestral = df_full.set_index('fecha')['ingreso'].resample('Q').sum()
variacion  = trimestral.pct_change() * 100
for fecha, total in trimestral.items():
    var = variacion[fecha]
    signo = '↑' if pd.notna(var) and var > 0 else '↓'
    var_str = f'{signo}{abs(var):.1f}%' if pd.notna(var) else 'base'
    print(f'  {fecha.strftime("%Y Q%q")}  ${total:>12,.0f}  {var_str}')

# Análisis 2
volatilidad = df_full.set_index('fecha')['ingreso'].resample('D').sum().resample('W').std()
print(volatilidad.dropna().tail(8).round(0))

# Análisis 3
cli_2023 = set(df_full[df_full['fecha'].dt.year == 2023]['cliente_id'])
cli_2024 = set(df_full[df_full['fecha'].dt.year == 2024]['cliente_id'])
churn = cli_2023 - cli_2024
print(f'Clientes en churn: {len(churn)} de {len(cli_2023)} ({len(churn)/len(cli_2023)*100:.1f}%)')

# Análisis 4
ticket_diario = df_full.set_index('fecha')['precio'].resample('D').mean()
ma7 = ticket_diario.rolling(7).mean()
print(ma7.dropna().tail(10).round(2))
```
</details>

### 🚀 Ejercicio libre 1 — Análisis de cohortes

Un **análisis de cohortes** agrupa usuarios por su fecha de primera compra y rastrea su comportamiento a lo largo del tiempo.

Usando `df_full`:

1. Determiná la **cohorte** de cada cliente: el mes de su primera compra
2. Calculá cuántos meses después de su primera compra volvió a comprar cada cliente
3. Construí la **tabla de retención**: filas = cohorte, columnas = mes 0,1,2,3..., valores = % de retención
4. Identificá qué cohorte tiene mejor retención a los 3 meses

**Pistas:**
- `df.groupby('cliente_id')['fecha'].min()` → primera compra por cliente
- Después de hacer el merge, calculá `(fecha_venta - primera_compra).dt.days // 30`
- `pivot_table` con `aggfunc=pd.Series.nunique` para contar clientes únicos

In [ ]:
import pandas as pd
import numpy as np

# 🚀 Tu análisis de cohortes aquí:

# Paso 1: Cohorte de cada cliente


# Paso 2: Mes de compra relativo


# Paso 3: Tabla de retención


# Paso 4: Mejor cohorte


---
## 🔢 Bloque 2: NumPy Avanzado

### 📘 Teoría

#### Broadcasting — reglas
NumPy puede operar arrays de distintas formas si son **compatibles**:
```
Regla: comparar dimensiones de derecha a izquierda
  - Si son iguales → OK
  - Si una es 1 → se 'expande'
  - Si son distintas y ninguna es 1 → ERROR

(3, 4)  +  (4,)   → (3, 4)  ✅  (se expande a (1,4) → (3,4))
(3, 4)  +  (3,)   → ERROR   ❌  (3 y 4 no son compatibles)
(3, 4)  +  (3,1)  → (3, 4)  ✅  (columna se expande a (3,4))
```

#### Álgebra lineal
```python
import numpy as np

A = np.array([[1,2],[3,4]])
np.linalg.det(A)          # determinante
np.linalg.inv(A)          # inversa
np.linalg.eig(A)          # valores y vectores propios
U, S, Vt = np.linalg.svd(A)  # descomposición SVD
```

#### Generación de números aleatorios
```python
rng = np.random.default_rng(seed=42)  # generador moderno
rng.normal(loc=0, scale=1, size=(100, 3))
rng.choice(array, size=50, replace=False)
rng.integers(0, 100, size=1000)
```

### 💡 Ejemplo resuelto 2 — NumPy para análisis cuantitativo

In [ ]:
import numpy as np

# ✅ NumPy avanzado: broadcasting, álgebra lineal, simulaciones

rng = np.random.default_rng(42)

# ─── Broadcasting avanzado ───
print('=== Broadcasting avanzado ===')

# Matriz de precios (5 productos × 3 tamaños)
precios = np.array([
    [100, 150, 200],
    [80,  120, 160],
    [200, 300, 400],
    [50,   75, 100],
    [300, 450, 600],
])  # shape (5, 3)

# Aplicar descuento por tamaño: [5%, 10%, 15%]
descuentos = np.array([0.05, 0.10, 0.15])  # shape (3,) → broadcast a (5,3)
precios_con_descuento = precios * (1 - descuentos)
print('Precios originales (5 prods × 3 tamaños):')
print(precios)
print('Con descuentos [5%, 10%, 15%]:')
print(precios_con_descuento.round(0))

# Centrar datos (restar media de cada columna)
datos = rng.normal(loc=[10, 100, 1000], scale=[2, 20, 200], size=(50, 3))
medias = datos.mean(axis=0)   # shape (3,)
stds   = datos.std(axis=0)    # shape (3,)
datos_norm = (datos - medias) / stds  # broadcasting: (50,3) - (3,) → OK
print(f'\nDatos normalizados — media: {datos_norm.mean(axis=0).round(10)}')
print(f'Std:   {datos_norm.std(axis=0).round(3)}')

# ─── Álgebra lineal: regresión lineal manual ───
print('\n=== Regresión lineal con álgebra matricial ===')

# Datos: relación entre superficie y precio de vivienda
n = 100
sup = rng.uniform(30, 200, n)
precio = 1200 * sup + rng.normal(0, 15000, n) + 20000

# Matriz de diseño X = [1, sup] (con intercepto)
X = np.column_stack([np.ones(n), sup])
y = precio

# Ecuación normal: β = (X'X)^-1 X'y
XtX_inv = np.linalg.inv(X.T @ X)
beta    = XtX_inv @ X.T @ y

print(f'  Intercepto (β₀): ${beta[0]:,.0f}')
print(f'  Pendiente   (β₁): ${beta[1]:,.0f} por m²')

# Comparar con numpy polyfit
beta_poly = np.polyfit(sup, precio, 1)
print(f'  Verificación polyfit: ${beta_poly[1]:,.0f} + ${beta_poly[0]:,.0f}/m²')

# R²
pred = X @ beta
ss_res = np.sum((y - pred)**2)
ss_tot = np.sum((y - y.mean())**2)
r2 = 1 - ss_res / ss_tot
print(f'  R²: {r2:.4f}')

# ─── SVD: compresión de imagen ───
print('\n=== SVD — Compresión de datos ===')

# Simular una 'imagen' como matriz de datos correlacionados
imagen = rng.normal(0, 1, (50, 50))
imagen += np.outer(np.sin(np.linspace(0, 3*np.pi, 50)),
                   np.cos(np.linspace(0, 3*np.pi, 50))) * 5

U, S, Vt = np.linalg.svd(imagen)
print(f'  Valores singulares (top 10): {S[:10].round(1)}')

# Reconstruir con k componentes
for k in [1, 5, 10, 25]:
    reconstruida = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    error = np.linalg.norm(imagen - reconstruida, 'fro') / np.linalg.norm(imagen, 'fro')
    compresion = k * (50 + 1 + 50) / (50 * 50) * 100
    print(f'  k={k:2}: error={error:.3f}, usa {compresion:.1f}% del espacio original')

# ─── Simulación Monte Carlo ───
print('\n=== Simulación Monte Carlo — VaR financiero ===')

# Simular retornos diarios de una cartera
retorno_anual = 0.12   # 12% anual esperado
vol_anual     = 0.25   # 25% volatilidad anual
precio_inicial = 100_000
dias = 252
n_sim = 100_000

retorno_diario = retorno_anual / dias
vol_diaria     = vol_anual / np.sqrt(dias)

# Simular n_sim trayectorias de días
retornos = rng.normal(retorno_diario, vol_diaria, (n_sim, dias))
precios_finales = precio_inicial * np.exp(retornos.sum(axis=1))

# Value at Risk (VaR) al 95%
var_95 = np.percentile(precios_finales, 5)
perdida_max_95 = precio_inicial - var_95

print(f'  Precio inicial:    ${precio_inicial:>10,.0f}')
print(f'  Precio esperado:   ${precios_finales.mean():>10,.0f}')
print(f'  Mediana:           ${np.median(precios_finales):>10,.0f}')
print(f'  Percentil 5%:      ${var_95:>10,.0f}')
print(f'  VaR 95% (pérdida max): ${perdida_max_95:>8,.0f}')
print(f'  % escenarios negativos: {(precios_finales < precio_inicial).mean()*100:.1f}%')

### ✏️ Ejercicio guiado 2 — Análisis estadístico con NumPy

Implementá estas operaciones estadísticas usando NumPy puro.

**Pistas:**
- La correlación de Pearson: `np.corrcoef(a, b)[0,1]`
- El percentil p: `np.percentile(arr, p)`
- La distancia euclidiana entre dos vectores: `np.linalg.norm(a - b)`

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# Datos de prueba
precios    = rng.uniform(50, 5000, 1000)
cantidades = rng.integers(1, 20, 1000)
ingresos   = precios * cantidades

# ✏️ Ejercicio 1: Calcular manualmente media, varianza y std
# Sin usar .mean(), .var(), .std() de numpy
print('=== Estadísticas manuales ===')
media = None  # sum(x) / n
varianza = None  # sum((x - media)²) / n
std = None  # sqrt(varianza)
print(f'  Media ingresos (manual): {media}')
print(f'  Verificación numpy:      {ingresos.mean():.2f}')

# ✏️ Ejercicio 2: Normalización min-max SIN sklearn
print('\n=== Normalización min-max ===')
precios_norm = None  # (precios - min) / (max - min)
print(f'  Rango normalizado: [{precios_norm.min() if precios_norm is not None else "?":.2f}, {precios_norm.max() if precios_norm is not None else "?":.2f}]')

# ✏️ Ejercicio 3: Correlación entre precio y cantidad
print('\n=== Correlación precio vs cantidad ===')
correlacion = None  # np.corrcoef(precios, cantidades)[0,1]
print(f'  Correlación: {correlacion}')

# ✏️ Ejercicio 4: Encontrar los 10 ingresos más altos sin sort completo
print('\n=== Top 10 ingresos ===')
# Pista: np.argpartition(arr, -10)[-10:]  — O(n) vs O(n log n) del sort
top10_idx = None
if top10_idx is not None:
    print(f'  Top 10: {np.sort(ingresos[top10_idx])[::-1].round(0)}')


<details>
<summary>👀 Ver solución</summary>

```python
# Ejercicio 1
n       = len(ingresos)
media   = np.sum(ingresos) / n
varianza= np.sum((ingresos - media)**2) / n
std     = np.sqrt(varianza)

# Ejercicio 2
precios_norm = (precios - precios.min()) / (precios.max() - precios.min())

# Ejercicio 3
correlacion = np.corrcoef(precios, cantidades)[0, 1]

# Ejercicio 4
top10_idx = np.argpartition(ingresos, -10)[-10:]
```
</details>

### 🚀 Ejercicio libre 2 — Optimización de cartera con NumPy

Implementá una simulación de **Markowitz (Teoría Moderna de Carteras)**:

1. Generá retornos diarios simulados para **5 activos** (250 días)
2. Simulá **10.000 carteras** con pesos aleatorios que sumen 1
3. Para cada cartera calculá:
   - Retorno esperado: `w @ retornos_anuales`
   - Riesgo (volatilidad): `sqrt(w @ cov_matrix @ w) * sqrt(252)`
   - Ratio Sharpe: `(retorno - tasa_libre_riesgo) / riesgo`
4. Identificá la **cartera de máximo Sharpe** y la **cartera de mínima varianza**
5. Mostrá los pesos óptimos de cada cartera

**Pistas:**
- `np.random.dirichlet(np.ones(5))` genera 5 pesos que suman 1
- `np.cov(retornos.T)` calcula la matriz de covarianza
- `np.argmax(sharpe_ratios)` encuentra el índice del máximo

In [ ]:
import numpy as np

# 🚀 Tu simulación de Markowitz aquí:

# 1. Retornos simulados:


# 2. Simular carteras:


# 3. Calcular métricas:


# 4. Carteras óptimas:


# 5. Mostrar resultados:


---
## 📈 Bloque 3: Matplotlib — Visualizaciones Profesionales

### 📘 Teoría

#### Anatomía de una figura
```
Figure (la hoja)
  └── Axes (el gráfico, puede haber varios)
        ├── Title
        ├── X/Y Label
        ├── X/Y Tick labels
        ├── Plot (line, bar, scatter...)
        └── Legend
```

#### API orientada a objetos (recomendada)
```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ax = axes[0, 0]
ax.plot(x, y, color='steelblue', linewidth=2, label='Ventas')
ax.set_title('Tendencia de ventas')
ax.set_xlabel('Fecha')
ax.set_ylabel('Monto ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grafico.png', dpi=150, bbox_inches='tight')
plt.show()
```

#### Tipos de gráficos y cuándo usarlos

| Tipo | Cuándo | Código |
|------|--------|---------|
| Line | Tendencias temporales | `ax.plot(x, y)` |
| Bar | Comparar categorías | `ax.bar(cats, valores)` |
| Scatter | Correlación entre dos vars | `ax.scatter(x, y)` |
| Histogram | Distribución de una var | `ax.hist(datos, bins=30)` |
| Boxplot | Distribución por grupos | `ax.boxplot([g1, g2])` |
| Heatmap | Matrices de correlación | `ax.imshow(corr)` |

### 💡 Ejemplo resuelto 3 — Dashboard de ventas con subplots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

# ✅ Dashboard completo con subplots y estilos profesionales

np.random.seed(42)

# Datos
fechas     = pd.date_range('2023-01-01', '2024-12-31', freq='D')
regiones   = ['AMBA','Centro','NOA','Cuyo','Patagonia']
categorias = ['Electrónica','Ropa','Hogar','Deportes']

n = len(fechas)
df = pd.DataFrame({
    'fecha':     fechas,
    'region':    np.random.choice(regiones, n),
    'categoria': np.random.choice(categorias, n),
    'ingreso':   np.random.exponential(scale=1500, size=n).round(2),
    'cantidad':  np.random.randint(1, 10, n),
})

# Agregar tendencia
tendencia = np.linspace(0, 500, n)
df['ingreso'] += tendencia

# Crear figura con GridSpec
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Dashboard de Ventas 2023-2024', fontsize=16, fontweight='bold', y=0.98)
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Gráfico 1: Tendencia mensual (ocupa 2 columnas) ──
ax1 = fig.add_subplot(gs[0, :2])
mensual = df.set_index('fecha')['ingreso'].resample('M').sum()
ma3 = mensual.rolling(3).mean()
ax1.fill_between(mensual.index, mensual.values, alpha=0.3, color='steelblue')
ax1.plot(mensual.index, mensual.values, color='steelblue', linewidth=1.5, label='Mensual')
ax1.plot(ma3.index, ma3.values, color='red', linewidth=2, linestyle='--', label='MA 3 meses')
ax1.set_title('Ingreso Mensual con Media Móvil')
ax1.set_ylabel('Ingreso ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Gráfico 2: Dona de categorías ──
ax2 = fig.add_subplot(gs[0, 2])
por_cat = df.groupby('categoria')['ingreso'].sum()
colors  = ['#2196F3','#4CAF50','#FF9800','#9C27B0']
wedges, texts, autotexts = ax2.pie(
    por_cat.values, labels=por_cat.index,
    autopct='%1.1f%%', colors=colors,
    wedgeprops={'edgecolor':'white','linewidth':2},
    pctdistance=0.8
)
for at in autotexts: at.set_fontsize(8)
ax2.set_title('Participación por Categoría')

# ── Gráfico 3: Barras horizontales por región ──
ax3 = fig.add_subplot(gs[1, 0])
por_region = df.groupby('region')['ingreso'].sum().sort_values()
bars = ax3.barh(por_region.index, por_region.values,
                color=['#4CAF50' if v == por_region.max() else '#2196F3'
                       for v in por_region.values])
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax3.set_title('Ingreso por Región')
ax3.set_xlabel('Ingreso Total')
# Anotar valores en las barras
for bar in bars:
    ax3.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
             f'${bar.get_width()/1000:.0f}k',
             va='center', fontsize=8)
ax3.grid(True, axis='x', alpha=0.3)

# ── Gráfico 4: Distribución de ingresos (histograma con KDE manual) ──
ax4 = fig.add_subplot(gs[1, 1])
ax4.hist(df['ingreso'], bins=50, color='steelblue', alpha=0.7, density=True, edgecolor='white')
# Percentiles
p25, p50, p75 = np.percentile(df['ingreso'], [25, 50, 75])
for p, label, color in [(p25,'P25','orange'),(p50,'Mediana','red'),(p75,'P75','green')]:
    ax4.axvline(p, color=color, linewidth=2, linestyle='--', label=f'{label}: ${p:.0f}')
ax4.set_title('Distribución de Ingresos')
ax4.set_xlabel('Ingreso ($)')
ax4.set_ylabel('Densidad')
ax4.legend(fontsize=8)

# ── Gráfico 5: Heatmap mes × día de semana ──
ax5 = fig.add_subplot(gs[1, 2])
df['mes']        = df['fecha'].dt.month
df['dia_semana'] = df['fecha'].dt.dayofweek
pivot = df.groupby(['dia_semana','mes'])['ingreso'].mean().unstack()
dias  = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
meses = ['E','F','M','A','M','J','J','A','S','O','N','D']
im = ax5.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax5.set_xticks(range(12))
ax5.set_xticklabels(meses, fontsize=8)
ax5.set_yticks(range(7))
ax5.set_yticklabels(dias, fontsize=8)
ax5.set_title('Ingreso Promedio\nMes × Día de Semana')
plt.colorbar(im, ax=ax5, fraction=0.046, pad=0.04)

plt.savefig('/mnt/user-data/outputs/dashboard_ventas.png', dpi=120, bbox_inches='tight')
print('✅ Dashboard guardado en outputs/dashboard_ventas.png')
print(f'   Figura: {fig.get_size_inches()} pulgadas, {fig.get_dpi()} DPI')
plt.close()

### ✏️ Ejercicio guiado 3 — Gráfico de comparación con anotaciones

Creá un gráfico que compare ventas online vs tienda con:
- Doble eje Y (uno para cada canal)
- Anotaciones en los picos
- Región sombreada entre las dos líneas

**Pistas:**
- `ax.twinx()` crea un segundo eje Y compartiendo el mismo eje X
- `ax.annotate('texto', xy=(x,y), xytext=(x+dx, y+dy), arrowprops=dict(...))`
- `ax.fill_between(x, y1, y2, alpha=0.1)` sombrea el área entre dos curvas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Datos mensuales por canal
np.random.seed(1)
meses = pd.date_range('2023-01-01', periods=24, freq='M')
online = np.random.normal(50000, 8000, 24) + np.linspace(0, 30000, 24)
tienda = np.random.normal(35000, 6000, 24) - np.linspace(0, 10000, 24)

fig, ax1 = plt.subplots(figsize=(12, 5))

# ✏️ Línea canal online en ax1:
# ax1.plot(meses, online, ...)

# ✏️ Crear ax2 = ax1.twinx() y graficar tienda:


# ✏️ Anotar el pico de online:
# idx_max = np.argmax(online)
# ax1.annotate(...)


# ✏️ Agregar fill_between entre online y tienda (en la misma escala):


plt.title('Comparación de canales 2023-2024')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/comparacion_canales.png', dpi=100, bbox_inches='tight')
print('✅ Guardado')
plt.close()


### 🚀 Ejercicio libre 3 — Tu propio dashboard

Creá un dashboard de al menos **6 gráficos** sobre el dataset que has estado analizando durante el curso. Requisitos:

1. Usá `GridSpec` para un layout personalizado (no solo 2×3)
2. Al menos un gráfico de línea con media móvil
3. Al menos un heatmap de correlación o temporalidad
4. Al menos una visualización de distribución (histograma o boxplot)
5. Anotaciones en al menos un gráfico (máximo, mínimo, o evento importante)
6. Título general y subtítulos individuales coherentes
7. Guardalo como PNG en `/mnt/user-data/outputs/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# 🚀 Tu dashboard aquí:

fig = plt.figure(figsize=(16, 12))
gs  = GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

# Gráfico 1:


# Gráfico 2:


# (continúa...)

plt.savefig('/mnt/user-data/outputs/mi_dashboard.png', dpi=120, bbox_inches='tight')
print('✅ Dashboard guardado')
plt.close()


---
## 🎨 Bloque 4: Seaborn — Visualizaciones Estadísticas

### 📘 Teoría

**Seaborn** construye sobre Matplotlib y está diseñado para visualizaciones estadísticas.  
Trabaja directamente con DataFrames de pandas.

#### Categorías de gráficos

| Categoría | Función | Cuándo |
|-----------|---------|--------|
| Distribución | `histplot`, `kdeplot`, `ecdfplot` | Ver forma de una variable |
| Relacional | `scatterplot`, `lineplot` | Relación entre dos variables |
| Categórico | `boxplot`, `violinplot`, `barplot` | Variable continua por categoría |
| Matrices | `heatmap`, `clustermap` | Correlaciones, matrices |
| Grid | `FacetGrid`, `pairplot` | Múltiples subplots automáticos |

#### Integración con Matplotlib
```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Seaborn puede recibir el 'ax' de matplotlib
sns.boxplot(data=df, x='categoria', y='precio', ax=axes[0])
sns.scatterplot(data=df, x='superficie', y='precio',
                hue='barrio', size='ambientes', ax=axes[1])
```

### 💡 Ejemplo resuelto 4 — Análisis estadístico visual con Seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ✅ Suite completa de visualizaciones estadísticas con Seaborn

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

# Dataset de viviendas
n = 600
barrios   = ['Palermo','Belgrano','San Telmo','Almagro','Caballito']
df = pd.DataFrame({
    'barrio':      np.random.choice(barrios, n, p=[0.25,0.2,0.1,0.25,0.2]),
    'ambientes':   np.random.choice([1,2,3,4], n, p=[0.1,0.35,0.4,0.15]),
    'superficie':  np.random.normal(70, 25, n).clip(20, 200),
    'antiguedad':  np.random.randint(0, 60, n),
    'cochera':     np.random.choice(['Sí','No'], n, p=[0.4,0.6]),
})
mult = {'Palermo':1.3,'Belgrano':1.2,'San Telmo':0.9,'Almagro':1.0,'Caballito':1.05}
df['precio'] = (
    1100*df['superficie'] + 4000*df['ambientes']
    + 10000*(df['cochera']=='Sí').astype(int)
    - 150*df['antiguedad'] + np.random.normal(0,8000,n)
) * df['barrio'].map(mult)
df['precio'] = df['precio'].clip(30000, 500000).round(-3)
df['precio_m2'] = (df['precio'] / df['superficie']).round(0)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Análisis de Mercado Inmobiliario', fontsize=14, fontweight='bold')

# ── 1. Distribución de precios por barrio (violin) ──
orden = df.groupby('barrio')['precio'].median().sort_values(ascending=False).index
sns.violinplot(data=df, x='barrio', y='precio', order=orden,
               palette='muted', inner='box', ax=axes[0,0])
axes[0,0].set_title('Distribución de precios por barrio')
axes[0,0].set_ylabel('Precio (USD)')
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[0,0].tick_params(axis='x', rotation=20)

# ── 2. Scatter superficie vs precio (coloreado por barrio) ──
sns.scatterplot(data=df, x='superficie', y='precio', hue='barrio',
                alpha=0.5, s=20, palette='Set1', ax=axes[0,1])
# Agregar línea de regresión
m, b = np.polyfit(df['superficie'], df['precio'], 1)
x_line = np.array([df['superficie'].min(), df['superficie'].max()])
axes[0,1].plot(x_line, m*x_line+b, 'k--', linewidth=2, label='Tendencia')
axes[0,1].set_title('Precio vs Superficie')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))
axes[0,1].legend(fontsize=7)

# ── 3. Barplot precio promedio por ambientes y cochera ──
sns.barplot(data=df, x='ambientes', y='precio', hue='cochera',
            palette={'Sí':'#2196F3','No':'#FF9800'}, ax=axes[0,2])
axes[0,2].set_title('Precio por ambientes y cochera')
axes[0,2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}k'))
axes[0,2].legend(title='Cochera')

# ── 4. Heatmap de correlaciones ──
numericas = df[['superficie','ambientes','antiguedad','precio','precio_m2']]
corr_matrix = numericas.corr()
mask = np.triu(np.ones_like(corr_matrix), k=1).astype(bool)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=mask, square=True,
            linewidths=0.5, ax=axes[1,0])
axes[1,0].set_title('Correlaciones')

# ── 5. KDE distribución precio/m² por barrio ──
for barrio in barrios:
    sub = df[df['barrio']==barrio]['precio_m2']
    sns.kdeplot(sub, label=barrio, ax=axes[1,1], fill=True, alpha=0.2)
axes[1,1].set_title('Distribución Precio/m² por barrio')
axes[1,1].set_xlabel('Precio/m² (USD)')
axes[1,1].legend(fontsize=8)

# ── 6. Boxplot antigüedad por barrio ──
sns.boxplot(data=df, x='barrio', y='antiguedad', order=orden,
            palette='pastel', ax=axes[1,2])
sns.stripplot(data=df, x='barrio', y='antiguedad', order=orden,
              color='black', alpha=0.1, size=2, jitter=True, ax=axes[1,2])
axes[1,2].set_title('Antigüedad por barrio')
axes[1,2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/analisis_inmobiliario.png', dpi=120, bbox_inches='tight')
print('✅ Análisis inmobiliario guardado')
plt.close()

### ✏️ Ejercicio guiado 4 — Pairplot y FacetGrid

Estas dos funciones generan múltiples subplots automáticamente.

**Pistas:**
- `sns.pairplot(df, hue='categoria', diag_kind='kde')` — todos contra todos
- `g = sns.FacetGrid(df, col='barrio', row='cochera', height=3)` — grid por categorías
- `g.map(sns.histplot, 'precio')` — aplica un gráfico a cada celda del grid

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# (Requiere df del ejemplo anterior)

# ✏️ Pairplot de variables numéricas coloreado por cochera
print('Generando pairplot...')
df_small = df.sample(200, random_state=42)  # muestra para velocidad
# g = sns.pairplot(df_small[['superficie','ambientes','precio','cochera']],
#                  hue='cochera', diag_kind='kde', plot_kws={'alpha':0.4})
# g.fig.suptitle('Pairplot de variables de vivienda', y=1.02)
# plt.savefig('/mnt/user-data/outputs/pairplot.png', dpi=80, bbox_inches='tight')
# plt.close()
print('  (descomentá el código para generar el pairplot)')

# ✏️ FacetGrid: histograma de precio por barrio
print('\nGenerando FacetGrid...')
# g = sns.FacetGrid(df, col='barrio', col_wrap=3, height=3, sharey=False)
# g.map(sns.histplot, 'precio', bins=20)
# g.set_titles('{col_name}')
# g.set_axis_labels('Precio (USD)', 'Frecuencia')
# plt.savefig('/mnt/user-data/outputs/facetgrid_barrios.png', dpi=80, bbox_inches='tight')
# plt.close()
print('  (descomentá el código para generar el FacetGrid)')

# ✏️ Catplot: violinplot de precio/m² por barrio y cochera
# sns.catplot(data=df, x='barrio', y='precio_m2', hue='cochera',
#             kind='violin', height=5, aspect=2)
# plt.savefig('/mnt/user-data/outputs/catplot.png', dpi=80, bbox_inches='tight')
# plt.close()

print('\nEjercicios listos para descomentar y ejecutar')


### 🚀 Ejercicio libre 4 — Historia visual de tus datos

Creá una **historia visual** en 4 gráficos seaborn que cuente una narrativa sobre tu dataset.

**La historia debe tener:**
1. **Contexto** — ¿qué distribución tienen los datos? (`violinplot` o `kdeplot`)
2. **Relaciones** — ¿qué variable predice mejor la variable objetivo? (`scatterplot` con regresión)
3. **Segmentación** — ¿cómo varía el patrón entre grupos? (`FacetGrid` o `catplot`)
4. **Correlaciones** — ¿qué variables se mueven juntas? (`heatmap`)

Cada gráfico debe tener:
- Título descriptivo (no 'Gráfico 1', sino algo que comunique el hallazgo)
- Etiquetas de ejes en español
- Una conclusión escrita debajo en Markdown

Guardá los 4 gráficos como PNGs en `/mnt/user-data/outputs/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# 🚀 Tu historia visual aquí:

# Gráfico 1 — Contexto:


# Gráfico 2 — Relaciones:


# Gráfico 3 — Segmentación:


# Gráfico 4 — Correlaciones:


## 📝 Hallazgos visuales

1. **Distribución:** 
2. **Relación clave:** 
3. **Diferencias entre grupos:** 
4. **Correlaciones destacadas:** 

*(Doble click para editar)*

---
## ✅ Resumen de la Semana 19-20

### Lo que aprendiste

| Tema | Conceptos clave |
|------|-----------------|
| Pandas avanzado | `merge`, `melt/pivot`, `resample`, ventanas, análisis de cohortes |
| NumPy avanzado | Broadcasting, álgebra lineal, SVD, Monte Carlo, Markowitz |
| Matplotlib | GridSpec, subplots, anotaciones, estilos, dashboard |
| Seaborn | Violin, scatter, heatmap, KDE, FacetGrid, pairplot |

### ➡️ Próximos pasos — Semana 21-22
- Machine Learning avanzado: regresión lineal con scikit-learn
- Clasificación con Random Forest
- SQL avanzado: subconsultas y funciones de ventana

---

### 📚 Recursos recomendados
- [Pandas — Time Series](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Matplotlib — Gallery](https://matplotlib.org/stable/gallery/index.html)
- [Seaborn — Tutorial](https://seaborn.pydata.org/tutorial.html)
- [NumPy — Linear Algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) — gratuito online